# 05. Prompting and Decode Sweeps

- Parent issue: `#21`
- Status: `active`
- Summary: Run sparse prompt/decode sweeps against the frozen eval harness, rank configurations, and promote one candidate through the golden regression gate.
        


## Audience and Why It Matters

This notebook is for experiment owners deciding whether inference-time prompting gains are large enough to justify downstream trajectory analysis (`#22`) and SFT bias decisions (`#25`).

The notebook stays intentionally strict: once sweep work starts, it refuses to run on synthetic data or guessed split files.
        


## Decision / Hypothesis

Hypothesis: a small prompt/decode sweep can beat the baseline run from notebook `04` without changing model weights, and the promoted configuration must still clear the golden regression gate.

Decision rules:
- Reuse `run_baseline_eval()` so sweep runs produce the same artifact contract as the baseline.
- Compare all sweep rows against an explicit `BASELINE_RUN_ID` from issue `#19`.
- Promote only a deterministic config snapshot with a stable `run_id` and a passing golden gate.
        


## Environment and Reproduction

- Python: 3.11+
- GPU: required for the real Nemotron sweep; the notebook raises if CUDA is unavailable.
- Run from repo root: `jupyter lab notebooks/05_prompting_and_decode_sweeps.ipynb`
- Required inputs: `data/eval/val.jsonl`, `data/eval/golden.jsonl`, and a compatible baseline run under `data/eval/runs/<BASELINE_RUN_ID>/`.
- Outputs:
  - `data/eval/runs/<run_id>/...` for every seeded run
  - `experiments/prompting_sweep_<date>.csv`
  - `docs/analysis/prompting_findings.md`

No synthetic fallback exists in this notebook. Missing split artifacts are a hard error by design.
        


In [ ]:
import os
import sys
import time
import platform
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Robust repo-root detection: walk upward until we find src/ + notebooks/.
_cwd = Path.cwd()
REPO_ROOT = _cwd
for _candidate in [_cwd] + list(_cwd.parents):
    if (_candidate / "src").is_dir() and (_candidate / "notebooks").is_dir():
        REPO_ROOT = _candidate
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation.config import make_run_config
from src.evaluation.golden_gate import evaluate_golden_gate, summarize_gate
from src.evaluation.prompt_sweeps import (
    DEFAULT_BEST_OF_N,
    DEFAULT_DECODE_GRID,
    DEFAULT_SEEDS,
    DEFAULT_STRATEGIES,
    aggregate_sweep_results,
    build_best_of_n_specs,
    build_run_id,
    build_sparse_sweep_specs,
    load_baseline_reference,
    load_required_splits,
    majority_vote,
    render_findings_markdown,
    stable_int_seed,
    validate_baseline_compatibility,
    write_aggregate_csv,
    write_findings_markdown,
)
from src.evaluation.runner import PredictRequest, PredictResponse, run_baseline_eval

print(f"Repo root : {REPO_ROOT}")
print(f"Python    : {sys.version.split()[0]}")
print(f"Platform  : {platform.platform()}")
print(f"CUDA      : {torch.cuda.is_available()}")
        


## Method and Outputs

### Sweep matrix
- Strategies: `zero-shot-cot`, `few-shot-cot`
- Decode grid: `(temperature, top_p)` in `{(0.6, 0.9), (0.6, 0.95), (1.0, 0.9), (1.0, 0.95)}`
- Seeds: `3` per config
- Best-of-N follow-up: `N in {8, 32}` using majority vote over sampled answers from the best sparse config

### Artifact policy
- Each seeded run writes the standard eval artifact set under `data/eval/runs/<run_id>/`.
- The aggregate CSV is one row per unique config and includes `accuracy_mean`, `accuracy_std`, and `delta_vs_baseline`.
- The findings markdown captures the ranked table, promotion decision, and golden gate verdict.
        


In [ ]:
DATE_STAMP = os.getenv(
    "PROMPT_SWEEP_DATE",
    datetime.now(timezone.utc).date().isoformat(),
)
MODEL_ID = os.getenv(
    "NEMOTRON_MODEL_ID",
    "nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16",
)
BASELINE_RUN_ID = os.getenv("PROMPT_SWEEP_BASELINE_RUN_ID", "baseline-stub-v1")
NORMALIZER_ID = "strip_v1"
MAX_NEW_TOKENS = 256
PROMOTION_SEED = DEFAULT_SEEDS[0]
ENABLE_WANDB = False
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "nemotron-prompting-sweep")

DECODE_GRID = DEFAULT_DECODE_GRID
SEEDS = DEFAULT_SEEDS
STRATEGIES = DEFAULT_STRATEGIES
BEST_OF_N_VALUES = DEFAULT_BEST_OF_N

CSV_OUTPUT_PATH = REPO_ROOT / "experiments" / f"prompting_sweep_{DATE_STAMP}.csv"
FINDINGS_OUTPUT_PATH = REPO_ROOT / "docs" / "analysis" / "prompting_findings.md"

print(f"Date stamp       : {DATE_STAMP}")
print(f"Model id         : {MODEL_ID}")
print(f"Baseline run id  : {BASELINE_RUN_ID}")
print(f"CSV output       : {CSV_OUTPUT_PATH.relative_to(REPO_ROOT)}")
print(f"Findings output  : {FINDINGS_OUTPUT_PATH.relative_to(REPO_ROOT)}")
        


In [ ]:
val_rows, golden_rows = load_required_splits(REPO_ROOT)
baseline = load_baseline_reference(REPO_ROOT, baseline_run_id=BASELINE_RUN_ID)
validate_baseline_compatibility(
    baseline=baseline,
    val_rows=val_rows,
    expected_model_id=MODEL_ID,
    expected_normalizer_id=NORMALIZER_ID,
)

DATASET_VERSION = val_rows[0].dataset_version
BASELINE_ACCURACY = float(baseline.summary["accuracy"])

print(f"Loaded val rows    : {len(val_rows)}")
print(f"Loaded golden rows : {len(golden_rows)}")
print(f"Dataset version    : {DATASET_VERSION}")
print(f"Baseline accuracy  : {BASELINE_ACCURACY:.4f}")
print(f"Baseline model     : {baseline.config.model_id}")
        


## Prompt Strategies

`zero-shot-cot` asks the model to reason carefully and then emit only the final answer. `few-shot-cot` adds three worked examples chosen to stay simple and off-benchmark.

The predictor extracts the final answer line before handing the result to `run_baseline_eval()`. That keeps scoring under the same `strip_v1` contract used by the baseline while leaving trajectory collection to notebook `06`.
        


In [ ]:
FEW_SHOT_EXAMPLES = [
    {
        "question": "Convert decimal 10 to binary.",
        "reasoning": "10 in decimal is 8 + 2, so the binary representation is 1010.",
        "answer": "1010",
    },
    {
        "question": "Solve for x: 3x - 6 = 9.",
        "reasoning": "Add 6 to both sides to get 3x = 15, then divide by 3.",
        "answer": "5",
    },
    {
        "question": "Decode ROT13: uryyb",
        "reasoning": "ROT13 shifts each letter by 13 places, so uryyb becomes hello.",
        "answer": "hello",
    },
]

SYSTEM_PROMPT = (
    "You solve short reasoning tasks. Work carefully, but end with a final line "
    "that contains only the answer with no label or extra commentary."
)


def render_prompt(problem_prompt: str, *, strategy: str) -> str:
    if strategy == "zero-shot-cot":
        return (
            "Solve the following task. Think step by step before deciding, "
            "then put only the final answer on the last line.

"
            f"Task:
{problem_prompt}"
        )
    if strategy == "few-shot-cot":
        blocks = []
        for i, example in enumerate(FEW_SHOT_EXAMPLES, start=1):
            blocks.append(
                f"Example {i}
"
                f"Task: {example['question']}
"
                f"Reasoning: {example['reasoning']}
"
                f"Answer: {example['answer']}"
            )
        blocks.append(
            "Now solve the next task. Think step by step, then put only the final "
            f"answer on the last line.

Task:
{problem_prompt}"
        )
        return "

".join(blocks)
    raise ValueError(f"Unknown strategy: {strategy!r}")


def render_model_input(tokenizer, *, strategy: str, problem_prompt: str) -> str:
    user_prompt = render_prompt(problem_prompt, strategy=strategy)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    return SYSTEM_PROMPT + "

" + user_prompt + "

Answer:
"


def extract_final_answer(raw_text: str) -> str:
    lines = [line.strip() for line in raw_text.splitlines() if line.strip()]
    if not lines:
        return ""
    last_line = lines[-1]
    lowered = last_line.lower()
    for prefix in ("final answer:", "answer:"):
        if lowered.startswith(prefix):
            return last_line.split(":", 1)[1].strip()
    return last_line
        


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "Notebook 05 requires a CUDA runtime for the real prompt sweep. "
        "Switch to a GPU environment after issue #18 split artifacts are available."
    )

torch.set_grad_enabled(False)
device = torch.device("cuda")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

INPUT_DEVICE = model.get_input_embeddings().weight.device
print(f"Model input device: {INPUT_DEVICE}")
print(f"Pad token id      : {tokenizer.pad_token_id}")
        


In [ ]:
def make_predictor(*, run_id: str, strategy: str, temperature: float, top_p: float, best_of_n: int = 1):
    if best_of_n < 1:
        raise ValueError(f"best_of_n must be >= 1, got {best_of_n}")

    def predictor(request: PredictRequest) -> PredictResponse:
        rendered_prompt = render_model_input(
            tokenizer,
            strategy=strategy,
            problem_prompt=request.prompt,
        )
        encoded = tokenizer(rendered_prompt, return_tensors="pt")
        encoded = {key: value.to(INPUT_DEVICE) for key, value in encoded.items()}
        prompt_tokens = int(encoded["input_ids"].shape[1])
        sample_count = best_of_n if best_of_n > 1 else 1
        do_sample = True
        generator = torch.Generator(device=str(INPUT_DEVICE))
        generator.manual_seed(
            stable_int_seed(run_id, request.example_id, request.seed, sample_count)
        )

        started = time.perf_counter()
        with torch.inference_mode():
            outputs = model.generate(
                **encoded,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=do_sample,
                temperature=temperature,
                top_p=top_p,
                num_return_sequences=sample_count,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                generator=generator,
            )
        elapsed_ms = (time.perf_counter() - started) * 1000.0

        input_length = int(encoded["input_ids"].shape[1])
        completion_ids = outputs[:, input_length:]
        decoded = tokenizer.batch_decode(completion_ids, skip_special_tokens=True)
        final_answers = [extract_final_answer(text) for text in decoded]
        prediction = majority_vote(final_answers)

        if tokenizer.pad_token_id is None:
            tokens_out = int(sum(seq.numel() for seq in completion_ids))
        else:
            tokens_out = int(
                sum((seq != tokenizer.pad_token_id).sum().item() for seq in completion_ids)
            )

        return PredictResponse(
            prediction=prediction,
            latency_ms=elapsed_ms,
            tokens_in=prompt_tokens * sample_count,
            tokens_out=tokens_out,
        )

    return predictor
        


In [ ]:
sparse_specs = build_sparse_sweep_specs(
    date_stamp=DATE_STAMP,
    strategies=STRATEGIES,
    decode_grid=DECODE_GRID,
    seeds=SEEDS,
)

sparse_rows = []
for spec in sparse_specs:
    predictor = make_predictor(
        run_id=spec.run_id,
        strategy=spec.strategy,
        temperature=spec.temperature,
        top_p=spec.top_p,
        best_of_n=spec.best_of_n,
    )
    run_config = make_run_config(
        run_id=spec.run_id,
        model_id=MODEL_ID,
        prompt_template_id=spec.strategy,
        normalizer_id=NORMALIZER_ID,
        seed=spec.seed,
        split="val",
        dataset_version=DATASET_VERSION,
        decode_config={
            "temperature": spec.temperature,
            "top_p": spec.top_p,
            "max_new_tokens": MAX_NEW_TOKENS,
            "do_sample": True,
            "best_of_n": spec.best_of_n,
            "output_parser": "final-answer-line-v1",
        },
    )

    started = time.perf_counter()
    result = run_baseline_eval(
        split_rows=val_rows,
        config=run_config,
        predictor=predictor,
        output_dir=REPO_ROOT / "data" / "eval" / "runs" / spec.run_id,
    )
    elapsed_seconds = time.perf_counter() - started
    sparse_rows.append(
        {
            "run_id": spec.run_id,
            "strategy": spec.strategy,
            "temperature": spec.temperature,
            "top_p": spec.top_p,
            "best_of_n": spec.best_of_n,
            "seed": spec.seed,
            "accuracy": float(result.summary["accuracy"]),
            "elapsed_seconds": elapsed_seconds,
            "correct": int(result.summary["correct"]),
            "total": int(result.summary["total"]),
        }
    )
    print(
        f"{spec.run_id}: accuracy={result.summary['accuracy']:.4f} "
        f"elapsed={elapsed_seconds:.1f}s"
    )

sparse_aggregate = aggregate_sweep_results(
    sparse_rows,
    baseline_accuracy=BASELINE_ACCURACY,
)
sparse_df = pd.DataFrame([row.as_csv_row() for row in sparse_aggregate])
assert len(sparse_df) >= 8, "Expected at least 8 unique sparse configs"
assert "delta_vs_baseline" in sparse_df.columns
sparse_df
        


In [ ]:
best_sparse = sparse_aggregate[0]
print("Best sparse config:")
print(best_sparse)

best_of_n_specs = build_best_of_n_specs(
    date_stamp=DATE_STAMP,
    strategy=best_sparse.strategy,
    temperature=best_sparse.temperature,
    top_p=best_sparse.top_p,
    seeds=SEEDS,
    best_of_n_values=BEST_OF_N_VALUES,
)

best_of_n_rows = []
for spec in best_of_n_specs:
    predictor = make_predictor(
        run_id=spec.run_id,
        strategy=spec.strategy,
        temperature=spec.temperature,
        top_p=spec.top_p,
        best_of_n=spec.best_of_n,
    )
    run_config = make_run_config(
        run_id=spec.run_id,
        model_id=MODEL_ID,
        prompt_template_id=spec.strategy,
        normalizer_id=NORMALIZER_ID,
        seed=spec.seed,
        split="val",
        dataset_version=DATASET_VERSION,
        decode_config={
            "temperature": spec.temperature,
            "top_p": spec.top_p,
            "max_new_tokens": MAX_NEW_TOKENS,
            "do_sample": True,
            "best_of_n": spec.best_of_n,
            "vote_rule": "majority-v1",
            "output_parser": "final-answer-line-v1",
        },
    )

    started = time.perf_counter()
    result = run_baseline_eval(
        split_rows=val_rows,
        config=run_config,
        predictor=predictor,
        output_dir=REPO_ROOT / "data" / "eval" / "runs" / spec.run_id,
    )
    elapsed_seconds = time.perf_counter() - started
    best_of_n_rows.append(
        {
            "run_id": spec.run_id,
            "strategy": spec.strategy,
            "temperature": spec.temperature,
            "top_p": spec.top_p,
            "best_of_n": spec.best_of_n,
            "seed": spec.seed,
            "accuracy": float(result.summary["accuracy"]),
            "elapsed_seconds": elapsed_seconds,
            "correct": int(result.summary["correct"]),
            "total": int(result.summary["total"]),
        }
    )
    print(
        f"{spec.run_id}: accuracy={result.summary['accuracy']:.4f} "
        f"elapsed={elapsed_seconds:.1f}s"
    )

all_rows = sparse_rows + best_of_n_rows
aggregate_rows = aggregate_sweep_results(
    all_rows,
    baseline_accuracy=BASELINE_ACCURACY,
)
aggregate_df = pd.DataFrame([row.as_csv_row() for row in aggregate_rows])
aggregate_df
        


In [ ]:
promoted = aggregate_rows[0]
promoted_run_id = build_run_id(
    date_stamp=DATE_STAMP,
    strategy=promoted.strategy,
    temperature=promoted.temperature,
    top_p=promoted.top_p,
    seed=PROMOTION_SEED,
    best_of_n=promoted.best_of_n,
) + "-golden"

promoted_predictor = make_predictor(
    run_id=promoted_run_id,
    strategy=promoted.strategy,
    temperature=promoted.temperature,
    top_p=promoted.top_p,
    best_of_n=promoted.best_of_n,
)

golden_config = make_run_config(
    run_id=promoted_run_id,
    model_id=MODEL_ID,
    prompt_template_id=promoted.strategy,
    normalizer_id=NORMALIZER_ID,
    seed=PROMOTION_SEED,
    split="golden",
    dataset_version=golden_rows[0].dataset_version,
    decode_config={
        "temperature": promoted.temperature,
        "top_p": promoted.top_p,
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": True,
        "best_of_n": promoted.best_of_n,
        "vote_rule": "majority-v1",
        "output_parser": "final-answer-line-v1",
    },
)

golden_result = run_baseline_eval(
    split_rows=golden_rows,
    config=golden_config,
    predictor=promoted_predictor,
    output_dir=REPO_ROOT / "data" / "eval" / "runs" / promoted_run_id,
)

golden_gate = evaluate_golden_gate(golden_result.records, golden_rows)
golden_summary = summarize_gate(golden_gate)
print(golden_summary)
        


In [ ]:
aggregate_csv_path = write_aggregate_csv(aggregate_rows, CSV_OUTPUT_PATH)
findings_notes = []
if not any(row.significant for row in aggregate_rows):
    findings_notes.append(
        "No configuration cleared the 2-sigma delta heuristic versus the selected baseline."
    )
findings_notes.append(
    "Raw model generations are reduced to final answers inside the predictor so the sweep remains comparable to the strip_v1 baseline."
)
findings_notes.append(
    "Use notebook 06 for trajectory capture once a promoted config has been selected."
)

findings_markdown = render_findings_markdown(
    date_stamp=DATE_STAMP,
    baseline=baseline,
    aggregate_rows=aggregate_rows,
    csv_path=aggregate_csv_path.relative_to(REPO_ROOT),
    golden_gate_passed=golden_gate.passed,
    golden_summary=golden_summary,
    promoted_run_id=promoted_run_id,
    notes=findings_notes,
)
write_findings_markdown(findings_markdown, FINDINGS_OUTPUT_PATH)

print(f"Wrote aggregate CSV : {aggregate_csv_path.relative_to(REPO_ROOT)}")
print(f"Wrote findings doc  : {FINDINGS_OUTPUT_PATH.relative_to(REPO_ROOT)}")
        


In [ ]:
assert len(aggregate_rows) >= 8, "Expected at least 8 unique config rows"
assert aggregate_df["accuracy_mean"].notna().all()
assert aggregate_df["accuracy_std"].notna().all()
assert aggregate_df["delta_vs_baseline"].notna().all()
assert "delta_vs_baseline" in aggregate_df.columns
print("Acceptance checks passed.")
        


## Results / Open Risks

Run this notebook end-to-end only after these prerequisites are true:
- `data/eval/val.jsonl` and `data/eval/golden.jsonl` exist from issue `#18`
- `BASELINE_RUN_ID` points at a notebook `04` run built on the same split artifact
- a GPU runtime is available for Nemotron inference

Open risks that still need judgment after the first full run:
- majority vote may improve accuracy but erase latency gains
- the best validation config may still fail the strict golden regression gate
- baseline drift is possible if `BASELINE_RUN_ID` points at a different dataset version or split size
        


## Sources

- [Prompt/decode notebook plan](../docs/planning/notebooks/05_prompting_and_decode_sweeps.md)
- [Baseline eval notebook](./04_baseline_eval_and_normalization.ipynb)
- [Eval runner](../src/evaluation/runner.py)
- [Golden gate](../src/evaluation/golden_gate.py)
- [Notebook registry](../docs/execution/NOTEBOOKS.md)
        
